# Smart-Shelf · 04 · Automated Margin Report (Deliverable 1)

**Deliverable 1 — flagship script.** Flag SKUs whose freight cost erodes margin. Three rules supported:\n- `loss` — true loss-makers (freight > price)\n- `heavy` — freight/price > 30% (default)\n- `bottom20` — worst 20% freight ratio within category

## Setup — auto-resolving paths

Run this cell first. It finds the project root automatically.

In [ ]:


from pathlib import Path
import os

# Find smart_shelf root by walking up from current working directory
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "scripts").exists() and (parent / "outputs").exists():
            return parent
        if parent.name == "smart_shelf":
            return parent
    # Fallback: assume notebook lives in smart_shelf/notebooks/
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"   # sibling of smart_shelf/
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

assert DATASET_DIR.exists(), f"Dataset folder not found at {DATASET_DIR}. Place Olist CSVs there."


## Configure the rule\n\nChange these variables to switch between rules.

In [ ]:
RULE = "heavy"          # one of: 'loss', 'heavy', 'bottom20'
THRESHOLD = 0.30        # freight-to-price ratio for 'heavy' rule
MIN_ORDERS = 5          # minimum order count per SKU to be considered

print(f"Rule       : {RULE}")
print(f"Threshold  : {THRESHOLD:.0%}")
print(f"Min orders : {MIN_ORDERS}")


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

m = pd.read_parquet(DATA_DIR / "master_orders.parquet")
print(f"Loaded: {len(m):,} rows")


## Step 1 — Aggregate to SKU level

In [ ]:
sku = (m.groupby(["product_id", "product_category_name_english"])
       .agg(orders            = ("order_id", "count"),
            total_revenue     = ("price", "sum"),
            total_freight     = ("freight_value", "sum"),
            avg_price         = ("price", "mean"),
            avg_freight       = ("freight_value", "mean"),
            avg_weight_g      = ("product_weight_g", "mean"),
            late_delivery_rate= ("is_late", "mean"),
            stockout_risk_rate= ("stockout_risk_flag", "mean"))
       .reset_index())

sku["freight_to_price_ratio"]   = sku["total_freight"] / sku["total_revenue"].replace(0, np.nan)
sku["margin_after_freight"]     = sku["total_revenue"] - sku["total_freight"]
sku["margin_pct_after_freight"] = (sku["margin_after_freight"] /
                                   sku["total_revenue"]).replace([np.inf, -np.inf], np.nan)

sku_full = sku.copy()
sku = sku[sku["orders"] >= MIN_ORDERS].copy()
print(f"SKUs analysed (≥ {MIN_ORDERS} orders): {len(sku):,} of {len(sku_full):,} total")


## Step 2 — Apply all three margin-leak rules

In [ ]:
# Rule A — true loss-makers
sku["flag_loss"] = (sku["total_freight"] > sku["total_revenue"]).astype(int)

# Rule B — heavy freight
sku["flag_heavy"] = (sku["freight_to_price_ratio"] > THRESHOLD).astype(int)

# Rule C — bottom-20% freight-to-price within category
sku["pctl_in_category"] = (sku.groupby("product_category_name_english")
                           ["freight_to_price_ratio"].rank(pct=True))
sku["flag_bottom20"] = (sku["pctl_in_category"] >= 0.80).astype(int)

flag_col = f"flag_{RULE}"
sku["MARGIN_LEAK_FLAG"] = sku[flag_col]

# Severity score for ranking
sku["severity_score"] = (sku["freight_to_price_ratio"].fillna(0)
                         * np.log1p(sku["orders"])
                         * (1 + sku["late_delivery_rate"].fillna(0)))
sku = sku.sort_values("severity_score", ascending=False)

flagged = sku[sku["MARGIN_LEAK_FLAG"] == 1]
print(f"Flagged: {len(flagged):,} SKUs ({len(flagged)/len(sku)*100:.1f}% of analysed)")


## Step 3 — Save outputs

In [ ]:
report_cols = ["product_id", "product_category_name_english", "orders",
               "total_revenue", "total_freight", "freight_to_price_ratio",
               "margin_after_freight", "margin_pct_after_freight",
               "late_delivery_rate", "stockout_risk_rate",
               "flag_loss", "flag_heavy", "flag_bottom20",
               "MARGIN_LEAK_FLAG", "severity_score"]
sku[report_cols].round(3).to_csv(OUTPUTS_DIR / "margin_leak_report.csv", index=False)

top50 = (flagged.head(50)[report_cols + ["avg_price", "avg_freight"]].round(2))
top50.to_csv(OUTPUTS_DIR / "top50_margin_leaks.csv", index=False)

cat_roll = (flagged.groupby("product_category_name_english")
            .agg(flagged_skus  = ("product_id", "nunique"),
                 freight_burden= ("total_freight", "sum"),
                 revenue_lost  = ("total_revenue", "sum"),
                 avg_ratio     = ("freight_to_price_ratio", "mean"))
            .sort_values("freight_burden", ascending=False)
            .round(2))
cat_roll.to_csv(OUTPUTS_DIR / "margin_leak_by_category.csv")

print("Saved 3 CSVs to outputs/")
cat_roll.head(10)


## Step 4 — Executive summary

In [ ]:
total_skus       = len(sku)
flagged_skus     = len(flagged)
flagged_pct      = flagged_skus / total_skus * 100 if total_skus else 0
revenue_at_risk  = flagged["total_revenue"].sum()
freight_burden   = flagged["total_freight"].sum()
avg_ratio_flag   = flagged["freight_to_price_ratio"].mean()

summary = f'''
========================================================================
OMNISTORE — AUTOMATED MARGIN-LEAK REPORT (Executive Summary)
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
========================================================================

RULE APPLIED            : {RULE}
THRESHOLD               : {THRESHOLD:.0%} freight-to-price ratio
MIN ORDERS PER SKU      : {MIN_ORDERS}

PORTFOLIO SCAN
  SKUs analysed         : {total_skus:>10,}
  SKUs flagged          : {flagged_skus:>10,}  ({flagged_pct:.1f} %)
  Revenue on flagged    : R$ {revenue_at_risk:>13,.0f}
  Freight burden        : R$ {freight_burden:>13,.0f}
  Avg freight ratio     : {avg_ratio_flag:.2%}

CROSS-CHECK ALL THREE RULES
  Loss-makers (freight > price)  : {sku['flag_loss'].sum():>6,} SKUs
  Heavy freight (>{THRESHOLD:.0%})        : {sku['flag_heavy'].sum():>6,} SKUs
  Bottom 20 % within category    : {sku['flag_bottom20'].sum():>6,} SKUs

TOP 5 MARGIN-LEAK CATEGORIES (by freight burden)
'''
for _, r in cat_roll.head(5).reset_index().iterrows():
    summary += (f"  - {r['product_category_name_english']:<30} "
                f"freight R$ {r['freight_burden']:>10,.0f}   "
                f"avg ratio {r['avg_ratio']:.0%}\n")

summary += f'''
TOP 5 SKU OFFENDERS (by severity score)
'''
for _, r in flagged.head(5).iterrows():
    summary += (f"  - {r['product_id'][:12]}…  "
                f"cat: {r['product_category_name_english']:<22}  "
                f"ratio {r['freight_to_price_ratio']:.0%}   "
                f"orders {int(r['orders']):>5}\n")

print(summary)
(OUTPUTS_DIR / "margin_leak_summary.txt").write_text(summary)


## Step 5 — Visualisations

In [ ]:
# Top 10 categories by freight burden
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(11, 6))
top10 = cat_roll.head(10).reset_index()
colors = sns.color_palette("Reds_r", len(top10))
bars = ax.barh(top10["product_category_name_english"][::-1],
               top10["freight_burden"][::-1], color=colors[::-1])
ax.set_xlabel("Total freight burden on flagged SKUs (R$)", fontsize=11)
ax.set_title("Top 10 Margin-Leak Categories by Freight Burden",
             fontsize=13, fontweight="bold", pad=12)
ax.bar_label(bars, fmt="R$%.0f", padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "margin_leak_top_categories.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# Distribution of freight-to-price ratio
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.histplot(sku["freight_to_price_ratio"].clip(0, 2), bins=60,
             color="#3b6ea8", ax=ax)
ax.axvline(THRESHOLD, color="red", linestyle="--", linewidth=2,
           label=f"Threshold ({THRESHOLD:.0%})")
ax.set_xlabel("Freight-to-price ratio", fontsize=11)
ax.set_ylabel("SKU count", fontsize=11)
ax.set_title("Distribution of Freight-to-Price Ratio Across SKUs",
             fontsize=13, fontweight="bold", pad=12)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "freight_ratio_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


✅ **Notebook 04 complete.** Move to `05_build_bullwhip_xlsx.ipynb`.